# Principles of Natural Language Processing Lab  
## Language Modelling: N-grams, N-gram Probabilities, Evaluation and Perplexity

**Course:** AME 5053 — Principles of Natural Language Processing Lab  
**Lab duration:** 3 hours  
**Mode:** Guided implementation + comparison + interpretation

### Learning outcomes
By the end of this lab, you should be able to:

1. construct unigram, bigram and trigram language models from a text corpus;
2. estimate N-gram probabilities using maximum-likelihood estimation (MLE);
3. compute the probability of a sentence under an N-gram model;
4. explain the effect of sentence-boundary symbols such as `<s>` and `</s>`;
5. evaluate a language model on unseen text using perplexity;
6. diagnose why an unsmoothed N-gram model can fail on unseen N-grams.

### What you must submit
Your notebook should contain:
- your predictions before running selected cells;
- completed implementations;
- at least one comparison between models;
- an error analysis using concrete examples;
- a short conclusion explaining what model behaviour you observed.

## 1. Corpus used in this lab

We will use a small, controlled corpus so that the counts can be inspected manually.

```text
students learn natural language processing
students learn machine learning
students study language models
language models predict words
language models assign probabilities
machine learning models learn patterns
natural language processing uses language models
```

The corpus is intentionally small. This makes the probability calculations transparent, but it will also expose an important limitation: **many valid word sequences will never occur in the training corpus**.

We will later split examples into **training** and **test** sentences so that perplexity is computed on data that the model did not simply memorize.

In [ ]:
from collections import Counter, defaultdict
import math
import pandas as pd

raw_sentences = [
    "students learn natural language processing",
    "students learn machine learning",
    "students study language models",
    "language models predict words",
    "language models assign probabilities",
    "machine learning models learn patterns",
    "natural language processing uses language models",
]

raw_sentences

## 1.1 Tokenize the corpus
Run the helper below and inspect the output.

In [ ]:
def tokenize(sentence):
    return sentence.lower().split()

tokenized = [tokenize(s) for s in raw_sentences]
tokenized

## 2. Sentence boundaries

A language model should know not only which words can follow other words, but also where a sentence can begin and end.

For a bigram model we will represent:

```text
students learn machine learning
```

as

```text
<s> students learn machine learning </s>
```

For a trigram model, we need enough history at the beginning. We therefore use two start symbols:

```text
<s> <s> students learn machine learning </s>
```

This convention allows the model to estimate probabilities for sentence beginnings as well as sentence endings.

### Prediction 1 — before coding

Without running any code, answer:

1. Which word do you expect to be the most frequent unigram?
2. Which bigram do you expect to have a relatively high probability?
3. Do you expect a trigram model to assign non-zero probability to more or fewer unseen test sentences than a bigram model? Why?

**Your prediction:**  
- Most frequent unigram: **`language`** (it appears repeatedly across the corpus).
- Likely high-probability bigram: **`language models`**, because it occurs several times.
- Bigram vs trigram on unseen text: **The trigram model should assign non-zero probability to fewer unseen sequences.** It uses a longer context, so there are more possible N-grams and more opportunities for a sequence to be absent from this small training corpus.


## 3. Build N-gram counts

For an N-gram model, we count neighbouring sequences of words.

For example, in:

```text
language models predict words
```

the bigrams are:

```text
(<s>, language)
(language, models)
(models, predict)
(predict, words)
(words, </s>)
```

The trigram model uses sequences of three tokens.

We will use these counts to estimate conditional probabilities.

In [ ]:
def add_boundaries(tokens, n):
    """Add sentence-boundary symbols appropriate for an N-gram order."""
    if n == 2:
        return ["<s>"] + list(tokens) + ["</s>"]
    elif n == 3:
        return ["<s>", "<s>"] + list(tokens) + ["</s>"]
    else:
        raise ValueError("This lab supports n=2 or n=3.")


In [ ]:
def make_ngrams(tokens, n):
    """Return all adjacent n-grams as tuples."""
    if n <= 0:
        raise ValueError("n must be positive")
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


In [ ]:
# Build unigram, bigram and trigram counts over the whole corpus.

unigram_counts = Counter(
    word
    for sentence in tokenized
    for word in sentence
)

bigram_counts = Counter(
    ngram
    for sentence in tokenized
    for ngram in make_ngrams(add_boundaries(sentence, 2), 2)
)

trigram_counts = Counter(
    ngram
    for sentence in tokenized
    for ngram in make_ngrams(add_boundaries(sentence, 3), 3)
)

unigram_counts, bigram_counts, trigram_counts


### Inspect the most frequent N-grams
Create tables showing the 10 most frequent unigrams, bigrams and trigrams.

In [ ]:
# Display the 10 most frequent items from each Counter.

unigram_table = pd.DataFrame(
    unigram_counts.most_common(10),
    columns=["unigram", "count"]
)

bigram_table = pd.DataFrame(
    bigram_counts.most_common(10),
    columns=["bigram", "count"]
)

trigram_table = pd.DataFrame(
    trigram_counts.most_common(10),
    columns=["trigram", "count"]
)

print("Top 10 unigrams")
display(unigram_table)

print("Top 10 bigrams")
display(bigram_table)

print("Top 10 trigrams")
display(trigram_table)


## 4. Maximum-likelihood estimation of N-gram probabilities

For a bigram model,

$$
P(w_i \mid w_{i-1})
=
\frac{C(w_{i-1}, w_i)}{C(w_{i-1})}
$$

For a trigram model,

$$
P(w_i \mid w_{i-2}, w_{i-1})
=
\frac{C(w_{i-2}, w_{i-1}, w_i)}
     {C(w_{i-2}, w_{i-1})}
$$

The key idea is simple: **among all occurrences of the history, how often was the next word the one we are asking about?**

This is a maximum-likelihood estimate because the probabilities are obtained directly from observed frequencies in the training corpus.

In [ ]:
def bigram_probability(w1, w2, bigram_counts, unigram_context_counts):
    """Compute P(w2 | w1) using maximum-likelihood estimation."""
    denominator = unigram_context_counts[w1]
    if denominator == 0:
        return 0.0
    return bigram_counts[(w1, w2)] / denominator


def trigram_probability(w1, w2, w3, trigram_counts, bigram_context_counts):
    """Compute P(w3 | w1, w2) using maximum-likelihood estimation."""
    denominator = bigram_context_counts[(w1, w2)]
    if denominator == 0:
        return 0.0
    return trigram_counts[(w1, w2, w3)] / denominator


### Probability checks
Compute and interpret the following:

- `P(learn | students)`
- `P(models | language)`
- `P(language | learn)`
- `P(models | study, language)`

In [ ]:
# Context counts are the denominators in the MLE formulas.

bigram_context_counts = Counter()
for (w1, w2), count in bigram_counts.items():
    bigram_context_counts[w1] += count

trigram_context_counts = Counter()
for (w1, w2, w3), count in trigram_counts.items():
    trigram_context_counts[(w1, w2)] += count

probability_checks = {
    "P(learn | students)": bigram_probability(
        "students", "learn", bigram_counts, bigram_context_counts
    ),
    "P(models | language)": bigram_probability(
        "language", "models", bigram_counts, bigram_context_counts
    ),
    "P(language | learn)": bigram_probability(
        "learn", "language", bigram_counts, bigram_context_counts
    ),
    "P(models | study, language)": trigram_probability(
        "study", "language", "models", trigram_counts, trigram_context_counts
    ),
}

pd.DataFrame(
    [{"probability": name, "value": value}
     for name, value in probability_checks.items()]
)


## 5. Sentence probability

Under a bigram model, the probability of a sentence is approximated using the Markov assumption:

$$P(w_1,\ldots,w_m) \approx P(w_1 \mid \text{<s>}) \prod_{i=2}^{m} P(w_i \mid w_{i-1}) P(\text{</s>} \mid w_m)$$

The trigram model conditions each word on the previous two tokens.

Because sentence probabilities are products of many values smaller than 1, they can become extremely small. In larger systems we therefore often work with **log probabilities**.

In [ ]:
def bigram_sentence_probability(sentence, bigram_counts, context_counts):
    """Compute sentence probability using an unsmoothed bigram model."""
    tokens = add_boundaries(tokenize(sentence), 2)
    probability = 1.0

    for w1, w2 in make_ngrams(tokens, 2):
        p = bigram_probability(w1, w2, bigram_counts, context_counts)
        probability *= p
        if p == 0:
            return 0.0

    return probability


def trigram_sentence_probability(sentence, trigram_counts, context_counts):
    """Compute sentence probability using an unsmoothed trigram model."""
    tokens = add_boundaries(tokenize(sentence), 3)
    probability = 1.0

    for w1, w2, w3 in make_ngrams(tokens, 3):
        p = trigram_probability(w1, w2, w3, trigram_counts, context_counts)
        probability *= p
        if p == 0:
            return 0.0

    return probability


### Compare sentence probabilities
Evaluate these two training-like sentences:

- `students learn machine learning`
- `language models assign probabilities`

Then explain why their probabilities differ.

In [ ]:
# Compute bigram and trigram sentence probabilities for the two sentences.

comparison_sentences = [
    "students learn machine learning",
    "language models learn patterns",
]

comparison_table = pd.DataFrame([
    {
        "sentence": sentence,
        "bigram_probability": bigram_sentence_probability(
            sentence, bigram_counts, bigram_context_counts
        ),
        "trigram_probability": trigram_sentence_probability(
            sentence, trigram_counts, trigram_context_counts
        ),
    }
    for sentence in comparison_sentences
])

comparison_table


## 6. Evaluation on unseen text

A useful language model should perform well on text that was **not used to estimate its probabilities**.

We will use:

### Test sentence A
`students learn language models`

### Test sentence B
`language models learn patterns`

### Test sentence C
`students predict probabilities`

Before computing anything, inspect the corpus and predict which test sentences may contain unseen bigrams or trigrams.

### Prediction 2 — unseen sequences

Before running the next section, list at least one bigram/trigram that you think is unseen in each test sentence.

**A. students learn language models**  
Prediction: **`learn language`** is likely unseen as a bigram; **`students learn language`** is likely unseen as a trigram.

**B. language models learn patterns**  
Prediction: the bigrams may be seen, but **`models learn patterns`** is likely unseen as a trigram.

**C. students predict probabilities**  
Prediction: **`students predict`** is likely unseen as a bigram; **`<s> students predict`** is likely unseen as a trigram.


## 7. Perplexity

Perplexity is a standard intrinsic evaluation measure for language models.

For a sequence containing \(N\) predicted tokens,

$$
PP(W)
=
P(W)^{-1/N}
$$

Equivalently, using log probabilities,

$$
PP(W)
=
\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}\log P(w_i \mid \text{history})
\right)
$$

Interpretation:

- **lower perplexity** means the model assigns higher probability to the observed test sequence;
- **higher perplexity** means the sequence is more surprising to the model;
- if an unsmoothed model encounters an N-gram with probability zero, the perplexity becomes infinite.

Perplexity should only be compared meaningfully when models are evaluated on the **same tokenization and same test data**.

In [ ]:
def bigram_perplexity(sentence, bigram_counts, context_counts):
    """Compute unsmoothed bigram perplexity using log probabilities."""
    tokens = add_boundaries(tokenize(sentence), 2)
    ngrams = make_ngrams(tokens, 2)

    log_probability_sum = 0.0

    for w1, w2 in ngrams:
        p = bigram_probability(w1, w2, bigram_counts, context_counts)
        if p == 0:
            return math.inf
        log_probability_sum += math.log(p)

    return math.exp(-log_probability_sum / len(ngrams))


def trigram_perplexity(sentence, trigram_counts, context_counts):
    """Compute unsmoothed trigram perplexity using log probabilities."""
    tokens = add_boundaries(tokenize(sentence), 3)
    ngrams = make_ngrams(tokens, 3)

    log_probability_sum = 0.0

    for w1, w2, w3 in ngrams:
        p = trigram_probability(w1, w2, w3, trigram_counts, context_counts)
        if p == 0:
            return math.inf
        log_probability_sum += math.log(p)

    return math.exp(-log_probability_sum / len(ngrams))


In [ ]:
test_sentences = [
    "students learn language models",
    "language models learn patterns",
    "students predict probabilities",
]

# Construct a DataFrame with sentence, bigram perplexity and trigram perplexity.

perplexity_table = pd.DataFrame([
    {
        "sentence": sentence,
        "bigram_perplexity": bigram_perplexity(
            sentence, bigram_counts, bigram_context_counts
        ),
        "trigram_perplexity": trigram_perplexity(
            sentence, trigram_counts, trigram_context_counts
        ),
    }
    for sentence in test_sentences
])

perplexity_table


## 8. Diagnose the model
For every test sentence with infinite perplexity, print the first unseen N-gram that causes the failure.

In [ ]:
def first_unseen_ngram(sentence, n, ngram_counts):
    """Return the first unseen n-gram, or None if all are observed."""
    tokens = add_boundaries(tokenize(sentence), n)
    for ngram in make_ngrams(tokens, n):
        if ngram not in ngram_counts:
            return ngram
    return None


# Apply the diagnostic to every test sentence for n=2 and n=3.

diagnostic_table = pd.DataFrame([
    {
        "sentence": sentence,
        "first_unseen_bigram": first_unseen_ngram(sentence, 2, bigram_counts),
        "first_unseen_trigram": first_unseen_ngram(sentence, 3, trigram_counts),
    }
    for sentence in test_sentences
])

diagnostic_table


## 9. Interpretation and error analysis

1. **Which model gave lower perplexity on the test sentences that both models could score?**  
   None of the three test sentences could be scored with finite perplexity by both models. The first and third have unseen bigrams, while the second has finite bigram perplexity but infinite trigram perplexity. Therefore, there is no fair finite-perplexity comparison between the two models on this particular test set.

2. **Did the higher-order model always perform better? Explain using the size of this corpus.**  
   No. The trigram model does not always perform better. This corpus is very small, so many valid three-word sequences are absent from training. The longer context makes the trigram model more data-hungry and causes zero probabilities for unseen trigrams.

3. **Identify one unseen bigram and one unseen trigram from the test set.**  
   An unseen bigram is **(`learn`, `language`)** from the first test sentence. An unseen trigram is **(`students`, `learn`, `language`)** from the first test sentence.

4. **Why does a zero probability create a serious problem when sentence probability is computed by multiplication?**  
   Sentence probability is the product of the probabilities of its N-grams. If even one N-gram has probability zero, the entire product becomes zero. In perplexity, log(0) is undefined (tends to negative infinity), so the resulting perplexity is treated as infinity.

5. **What would you expect to happen if the training corpus became much larger?**  
   A larger training corpus should contain more observed N-grams, reducing the number of zero-probability events. Trigram estimates would generally become more reliable, although unseen sequences can still occur. Smoothing is another standard way to handle unseen N-grams.


## 10. Viva / discussion questions

1. **Why is an N-gram language model called a probabilistic model?**  
   Because it assigns probabilities to words or word sequences based on their observed frequencies in a corpus.

2. **What is the Markov assumption in a bigram model?**  
   The bigram model assumes that the probability of the current word depends only on the immediately preceding word:  
   \(P(w_i \mid w_1,\ldots,w_{i-1}) \approx P(w_i \mid w_{i-1})\).

3. **Why are sentence boundary symbols useful?**  
   `<s>` marks where a sentence starts and `</s>` marks where it ends. They allow the model to learn beginning- and ending-of-sentence patterns and include sentence boundaries in probability calculations.

4. **Why can a trigram model be more data-hungry than a bigram model?**  
   A trigram conditions on two previous words, so it has many more possible contexts and requires more training examples to estimate reliable probabilities.

5. **Why is perplexity usually preferred over raw sentence probability when comparing sequences of different lengths?**  
   Raw sentence probability tends to become smaller as the sentence gets longer because more probabilities are multiplied. Perplexity normalizes by the number of predicted tokens, making comparisons across different sequence lengths more meaningful.

6. **Why can an unsmoothed N-gram model assign infinite perplexity to a perfectly grammatical sentence?**  
   The sentence may contain an N-gram that never appeared in the training corpus. Maximum-likelihood estimation then gives that N-gram probability zero, which makes the sentence probability zero and the perplexity infinite.
